# Übungs-Arbeitspaket AP02: Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse

*Selbst erstellte Übung im Stil des Probe-LN (fiktiver Shop «LaptopPlatz»). Nicht offiziell, Punkte nur zur Selbsteinschätzung.*

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

| Teil | Thema | Aufgaben | Punkte |
|---|---|---|---|
| A | Konzeptfragen: Datenaufbereitung & explorative Datenanalyse | A1–A3 | 12 |
| B | Fehleranalyse Python-Code (Shop-Daten) | B1–B2 | 16 |
| C | Python-Code schreiben: Web Scraping, Datenaufbereitung & EDA | C1–C3 | 12 |
| **Total** | | **8 Aufgaben** | **40** |

### Bewertungsschema

| Kategorie | Beschreibung | Bewertung | 4 Pkt | 8 Pkt |
|---|---|---|---|---|
| Korrekt | Vollständig und fachlich einwandfrei; Code liefert das richtige Ergebnis | 100% | 4 | 8 |
| Minimale Mängel | Kleine Fehler (Sortierung, Rundung, Achsenbeschriftung) oder Begründung ungenau/unvollständig | 75% | 3 | 6 |
| Mittlere Mängel | Grössere Fehler (fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. fachlich falsch | 50% | 2 | 4 |
| Gravierende Mängel | Ansatz erkennbar, Lösung im Kern falsch (z.B. falscher Join-Typ, falsche Tabelle) | 25% | 1 | 2 |
| Nicht lauffähig | Antwort fehlt, Code nicht lauffähig oder kein Bezug zur Aufgabe | 0% | 0 | 0 |

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zu den Daten

Alle Daten stammen vom **fiktiven** Schweizer Online-Shop «LaptopPlatz» und liegen **lokal** im gleichen Verzeichnis wie dieses Notebook.

| Datei | Format | Inhalt |
|---|---|---|
| `html/shop_page_1.html` … `shop_page_3.html` | HTML | Gespeicherte Produktlisten-Seiten (3 Seiten mit Paginierung) |
| `laptops_clean.csv` | CSV | Aufbereitete Shop-Daten: Formate vereinheitlicht, Duplikate entfernt — **inhaltlich nicht geprüft** |
| `lagerbestand.xlsx` | Excel | Lagerliste des Zentrallagers (Sheets «Info» und «bestand») |

**Spalten in `laptops_clean.csv`:**

| Spalte | Beschreibung |
|---|---|
| `product_id` | Artikelnummer des Shops |
| `brand`, `model`, `color` | Marke, Modell, Farbe |
| `ram_gb`, `storage_gb` | Arbeitsspeicher und SSD in GB |
| `screen_inch` | Bildschirmdiagonale in Zoll |
| `weight_kg` | Gewicht in kg (Herstellerangabe) |
| `battery_h` | Akkulaufzeit in Stunden (Herstellerangabe) |
| `price_chf` | Preis in ganzen CHF (leer bei «Auf Anfrage») |

**Antworten auf Konzeptfragen** und **Begründungen** tragen Sie direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import re
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Get current working directory
print(os.getcwd())

/workspaces/DS_HS2026_LNW/claude/AP02 Claude


### Produktlisten-Seiten scrapen → `df_raw`

Der folgende Code liest die drei gespeicherten Seiten ein. Jede Produktkarte (`<div class="product-card">`) wird zu einer Zeile in `df_raw`. Die Werte werden **als Text** übernommen (Spalten mit Endung `_raw`).

In [2]:
# Initialize list for the extracted rows
rows = []

# Loop over the three listing pages
for page in [1, 2, 3]:

    # Read html from file
    with open(f'html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        html_content = file.read()

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Locate the product cards and extract values
    for card in soup.select('div.product-card'):
        row_data = [card['data-product-id'],
                    card.select('span.brand')[0].get_text().replace('\xa0', ' ').strip(),
                    card.select('span.model')[0].get_text().replace('\xa0', ' ').strip(),
                    card.select('li.ram')[0].get_text().replace('\xa0', ' ').strip(),
                    card.select('li.storage')[0].get_text().replace('\xa0', ' ').strip(),
                    card.select('li.color')[0].get_text().replace('\xa0', ' ').strip(),
                    card.select('span.price')[0].get_text().replace('\xa0', ' ').strip()]
        rows.append(row_data)

# Create a DataFrame
df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'ram_raw',
                                     'storage_raw', 'color', 'price_raw'])

# Show dimensions and first rows
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

Number of rows and columns: (130, 7)


,product_id,brand,model,ram_raw,storage_raw,color,price_raw
0,LP-20001,Dell,XPS 13,32 GB RAM,1 TB SSD,Graphit,CHF 2'149.–
1,LP-20002,ASUS,Zenbook 14 OLED,16 GB RAM,1 TB SSD,Blau,CHF 1'399.–
2,LP-20003,ASUS,ExpertBook B9,32 GB RAM,1 TB SSD,Schwarz,CHF 2'649.–
3,LP-20004,ASUS,ExpertBook B9,16 GB RAM,512 GB SSD,Schwarz,CHF 2'149.–
4,LP-20005,Microsoft,"Surface Laptop 7 13.8""",32 GB RAM,1 TB SSD,Saphir,CHF 1'949.–


### Aufbereitete Shop-Daten laden → `df_clean`

In [3]:
# Import data from csv to pandas dataframe named 'df_clean'
df_clean = pd.read_csv('laptops_clean.csv')

# Show dimensions and first rows
print('Number of rows and columns:', df_clean.shape)
df_clean.head()

Number of rows and columns: (120, 10)


,product_id,brand,model,ram_gb,storage_gb,color,screen_inch,weight_kg,battery_h,price_chf
0,LP-20001,Dell,XPS 13,32,1024,Graphit,13.4,1.19,14,2149.0
1,LP-20002,ASUS,Zenbook 14 OLED,16,1024,Blau,14.0,1.20,16,1399.0
2,LP-20003,ASUS,ExpertBook B9,32,1024,Schwarz,14.0,0.99,17,2649.0
3,LP-20004,ASUS,ExpertBook B9,16,512,Schwarz,14.0,0.99,17,2149.0
4,LP-20005,Microsoft,"Surface Laptop 7 13.8""",32,1024,Saphir,13.8,1.34,16,1949.0


> **Hinweis:** Alle Aufgaben sind voneinander unabhängig. Verändern Sie `df_raw` und `df_clean` nicht direkt, sondern arbeiten Sie in jeder Aufgabe mit einem eigenen Data Frame (z.B. `df_B1`, `df_C2`).

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zu den Shop-Daten von «LaptopPlatz» werden nicht gewertet.*

### A1 (4 Punkte)

In `laptops_clean.csv` fehlt bei allen 5 Geräten der Marke **Medion** das Gewicht (`weight_kg`). Laut Shop-Betreiber veröffentlicht Medion für diese Modelle keine Gewichtsangaben. Bei allen anderen Marken ist das Gewicht vollständig erfasst. Medion führt im Sortiment vor allem 15.6-Zoll- und Gaming-Geräte.

a) Um welchen Mechanismus fehlender Werte handelt es sich (MCAR, MAR oder MNAR)? Begründen Sie.

b) Eine Analystin entfernt die 5 Geräte mit `dropna()` und berichtet das durchschnittliche Gewicht **aller Laptops** im Sortiment. Welchen Effekt kann das auf das Ergebnis haben? Begründen Sie.

**Ihre Antwort:**

a) MAR. Ob das Gewicht fehlt, hängt vom Lieferanten ab. Medion gibt generell nicht das Gewicht an. Die fehlenden Werte sind also systematisch die hohen Preise.

b) Mit dropna() werden Geräte entfernt. Das berichtete Durchschnittsgewicht ist deshalb ungenau (systematisch verzerrt). Hier weiss man leider nicht ob nach unten oder oben verzerrt. Jedoch sind die Zusammenhänge unverzerrt (dieser Lieferant hat nie ein Gewicht).

### A2 (4 Punkte)

Für die Spalte `battery_h` (Akkulaufzeit in Stunden) liefert pandas folgendes Ergebnis:

```
df_clean['battery_h'].describe()          df_clean['battery_h'].skew()
count    120.00                            -0.41
mean      12.82
std        4.92
min        3.00
25%        8.00
50%       14.00
75%       17.00
max       21.00
```

Beschreiben Sie die Verteilung der Akkulaufzeiten anhand dieser Kennzahlen. Welches Lagemass eignet sich besser, um die «typische» Akkulaufzeit zu berichten, und warum? Nennen Sie eine mögliche inhaltliche Erklärung für die Form der Verteilung und eine geeignete Grafik, um Ihre Einschätzung zu überprüfen.

**Ihre Antwort:**

mean 12.82, median 14
median>mean -> linksschief
-0.41 skew-> linksschief

Anhand der Kennzahlen sieht man dass die Mehrheit der Geräte eine Akkulaufzeit von 3h-17h hat, wobei der Durchschnitt bei 12.82h liegt. der Median liegt allerdings bei 14, was bedeutet dass es eher Ausreisser im unteren Ende gibt (min=3h). die Skewness beschreibt auch ein linksschiefes bild der Verteilung der Akkulaufzeiten pro GErät. 

Eine geeignete Grafik für die Schiefe wäre beispielsweise ein Histogramm. Für die Ausreisser wäre ein Boxplott nützlich, damit erkenntlich wird wie sich die Daten verteilen.

lagemass= median
boxplott oder scatter

### A3 (4 Punkte)

Ein Kollege möchte `price_chf` für ein Regressionsmodell vorbereiten. Er wendet eine **Min-Max-Skalierung** an und schreibt:

```python
df['price_scaled'] = (df['price_chf'] - df['price_chf'].min()) / (df['price_chf'].max() - df['price_chf'].min())
```

*«Jetzt liegen alle Preise zwischen 0 und 1. Damit ist die Verteilung symmetrisch und Ausreisser spielen keine Rolle mehr.»*

Nennen und begründen Sie **zwei** fachliche Einwände gegen diese Aussage. Nennen Sie zudem eine Methode, die für das genannte Ziel besser geeignet wäre.

**Ihre Antwort:**

Ein erheblicher Nachteil ist die Empfindlichkeit gegenüber Ausreißern von min-max. Da die Skalierung auf den Minimal- und Maximalwerten des Preises basiert, kann das Vorhandensein von Ausreißern den Skalierungsprozess verzerren, was zu einem komprimierten Bereich für die Mehrheit der Datenpunkte führt. Dies kann zu einem Informationsverlust führen. Geeigneter wäre eine standartisierte Max-Absolute scaling. Oder gar Robust Scaling. 

---
## Teil B – Fehleranalyse Python-Code (16 Punkte)

*Jeder Code läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Ergebnis. Die tatsächlichen Ergebnisse sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Kennzahlen für das Shop-Sortiment aus den gescrapten Rohdaten `df_raw`: Anzahl Geräte, Anzahl Geräte ohne Preisangabe, Median- und Durchschnittspreis in **ganzen CHF** (Rappenbeträge werden abgeschnitten, z.B. `1348.95` → `1348`).

```python
# Select columns for the analysis
df_B1 = df_raw[['product_id', 'brand', 'model', 'price_raw']]

# Extract numerical values from price_raw (note the 'join' function)
price = []
for i in df_B1['price_raw']:
    d1 = re.findall(r"\d+", i)
    try:
        d2 = int(''.join(d1))
    except:
        d2 = None
    price.append(d2)

# Save as new variable in the pandas data frame
df_B1['price_chf'] = pd.Series(price, dtype="Int64")

# Show key figures
print('Anzahl Geräte:            ', df_B1.shape[0])
print('Anzahl ohne Preisangabe:  ', df_B1['price_chf'].isnull().sum())
print('Median-Preis (CHF):       ', df_B1['price_chf'].median())
print('Durchschnittspreis (CHF): ', round(df_B1['price_chf'].mean(), 2))
```

Tatsächliches Ergebnis:

```
Anzahl Geräte:             130
Anzahl ohne Preisangabe:   5
Median-Preis (CHF):        1749.0
Durchschnittspreis (CHF):  40090.07
```

Auf jeder Shop-Seite steht: **«120 Artikel»**. Beispiele für Werte in `price_raw`: `CHF 849.–`, `CHF 1'249.–`, `CHF 1'348.95`, `Auf Anfrage`.

**Aufgabe:** Schreiben Sie den korrigierten Code in die Zelle «Korrigierter Code», sodass die Kennzahlen stimmen. Begründen Sie jede Änderung gegenüber dem ursprünglichen Code.

**Begründung der Änderungen:**

es sind 10 Geräte weniger angezeigt, als die  im shop verfügbar wären. diese müssen hinzugefügt werden.

zudem fehlt bei 5 die preisangabe. 

Die tausenderzeichen müssen entfernt werden und der mean preis muss korrigiert werden da der durchschnittspreis offenbar stark verzerrt wird. Dies aufgrund des aktuellen REGEX. dieses fügt die Rappenzahlen als normale Zahlen hinten an. Daher wie gem aufgabenstelluung, werden die rappen neu ignoriert. 

**Korrigierter Code:**

In [255]:
# Select columns for the analysis
df_B1 = df_raw[['product_id', 'brand', 'model', 'price_raw']]
df_B1 = df_B1.drop_duplicates().reset_index(drop=True)

# Extract price = []
price=[]
for i in df_B1['price_raw']:
    i = i.replace("'", "")     
    d1 = re.findall(r"\d+", i)
    try:
        d2 = int(d1[0])                   
    except:
        d2 = None
    price.append(d2)
    

# Save as new variable in the pandas data frame
df_B1['price_chf'] = pd.Series(price, dtype="Int64")

# Show key figures
print('Anzahl Geräte:            ', df_B1.shape[0])
print('Anzahl ohne Preisangabe:  ', df_B1['price_chf'].isnull().sum())
print('Median-Preis (CHF):       ', (round(df_B1['price_chf'].median(),)))
print('Durchschnittspreis (CHF): ', (round(df_B1['price_chf'].mean(),2)))

Anzahl Geräte:             120
Anzahl ohne Preisangabe:   5
Median-Preis (CHF):        1598
Durchschnittspreis (CHF):  1600.08


### B2 (8 Punkte)

**Ziel:** Herausfinden, welche Marke im Sortiment die **leichtesten** Laptops hat (auf Basis von `df_clean`).

```python
# Select columns for the analysis
df_B2 = df_clean[['brand', 'weight_kg']]

# Weight per brand
df_B2.groupby(['brand']).mean().round(2).sort_values(by='weight_kg', ascending=True)
```

Tatsächliches Ergebnis:

```
           weight_kg
brand
ASUS            1.40
Microsoft       1.42
Lenovo          1.52
Apple           1.57
Acer            1.74
Dell            1.75
HP              1.76
LG              1.99
Medion           NaN
```

Der Einkaufsleiter ist skeptisch: *«LG verkauft die gram-Serie als besonders leichte Laptops — dass LG am schwersten sein soll, kann nicht stimmen.»*

**Aufgabe:** Untersuchen Sie die Daten mit geeigneten EDA-Methoden, finden Sie die Ursache für das irreführende Ergebnis und schreiben Sie den korrigierten Code in die Zelle «Korrigierter Code». Begründen Sie jede Änderung und gehen Sie auf die Zeile «Medion NaN» ein.

**Begründung der Änderungen:**

[Hier eintragen]

**Korrigierter Code:**

In [256]:
# Korrigierten Code unter den Kommentaren eintragen

# Select columns for the analysis

# Weight per brand


---
## Teil C – Python-Code schreiben (12 Punkte)

Schreiben Sie Python-Code und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Auf den Produktlisten-Seiten wird die Kundenbewertung als Sterne angezeigt. Ein Ausschnitt aus `html/shop_page_1.html`:

```html
<div class="product-card" data-product-id="LP-20001">
  <a class="product-link" href="/laptop/LP-20001">
    <h2 class="product-title"><span class="brand">Dell</span> <span class="model">XPS 13</span></h2>
  </a>
  ...
  <div class="rating" data-rating="4.4">★★★★☆ <span class="count">132 Bewertungen</span></div>
  ...
</div>
```

Geräte ohne Bewertung haben statt `<div class="rating">` ein `<span class="no-rating">Noch keine Bewertungen</span>`.

Ergänzen Sie die drei mit `# TODO` markierten Stellen im Gerüst, sodass `df_C1` für **Seite 1** pro Gerät `model`, `rating` (als Zahl) und `url` enthält, und zeigen Sie die **5 bestbewerteten** Geräte (absteigend sortiert).

Begründen Sie, warum Sie die Bewertung so auslesen und wie Sie Geräte ohne Bewertung behandeln.

**Begründung der Vorgehensweise:**

[Hier eintragen]

**Python-Code:**

In [257]:
# Read html from file
with open('html/shop_page_1.html', 'r', encoding='utf-8') as file:
    html_content = file.read()

# Parse the HTML content
soup = BeautifulSoup(html_content, 'html.parser')

# Extract model, rating and url per product card
rows = []
for card in soup.select('div.product-card'):
    model = card.select('span.model')[0].get_text().strip()

    # TODO: Rating as number (None if the device has no rating)
    rating = ...

    # TODO: Link of the product (href)
    url = ...

    rows.append([model, rating, url])

# Create a DataFrame
df_C1 = pd.DataFrame(rows, columns=['model', 'rating', 'url'])

# TODO: Show the 5 best-rated devices (descending)


### C2 (4 Punkte)

Teilen Sie die Geräte in `df_clean` anhand von `screen_inch` mit `pd.cut()` in **drei feste Grössenklassen** ein (neue Spalte `groesse`):

| Klasse | Bereich |
|---|---|
| `kompakt` | bis und mit 14 Zoll |
| `standard` | über 14 bis und mit 15.6 Zoll |
| `gross` | über 15.6 Zoll |

Erstellen Sie anschliessend mit `pd.pivot_table()` eine Tabelle mit dem **Median-Preis** und der **Anzahl Geräte mit Preis** pro Grössenklasse (gerundet auf 0 Dezimalstellen).

Begründen Sie, warum hier `pd.cut()` und nicht `pd.qcut()` passt, und in welche Klasse ein Gerät mit genau 14.0 Zoll fällt.

**Begründung der Vorgehensweise:**

[Hier eintragen]

**Python-Code:**

In [258]:
# Code hier eintragen


### C3 (4 Punkte)

Lesen Sie den Lagerbestand aus dem Sheet **«bestand»** der Datei `lagerbestand.xlsx` ein und verbinden Sie ihn mit `.merge()` über `product_id` mit `df_clean`. Laut Sheet «Info» gilt: **Geräte ohne Eintrag in der Lagerliste sind nicht an Lager.**

Berechnen Sie pro Marke die **Anzahl Geräte im Sortiment** und den **Anteil der Geräte, die an Lager sind** (`lager_stueck` > 0), in Prozent und gerundet auf 1 Dezimalstelle. Sortieren Sie absteigend nach dem Anteil und stellen Sie den Anteil als **horizontales Balkendiagramm** mit beschrifteten Achsen dar.

Begründen Sie, welchen Join-Typ Sie gewählt haben und wie Sie die Geräte ohne Eintrag in der Lagerliste behandeln.

**Begründung der Vorgehensweise:**

[Hier eintragen]

**Python-Code:**

In [259]:
# Code hier eintragen


---
### Jupyter notebook --footer info--

In [260]:
import os, platform
from platform import python_version
from datetime import datetime
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-11-07 11:46:36
Python Version: 3.12.11
-----------------------------------
